# 🎨 БЕСПЛАТНЫЙ Генератор Картинок для Флешкарточек
## Stable Diffusion - Локальная генерация БЕЗ API ключей

✅ **Полностью бесплатно**
✅ **Без регистрации и ключей**
✅ **Работает в Google Colab**
✅ **Высокое качество изображений**

## 📦 Шаг 1: Установка зависимостей

⚠️ **Важно**: Выберите ОДИН из вариантов в зависимости от вашей системы

In [ ]:
# Для Google Colab - выполните это:
import subprocess
import sys

print("📦 Установка зависимостей...")
print("(Это займет 2-3 минуты в первый раз)\n")

# ВАЖНО: сначала фиксируем версию sympy — иначе часто возникает
# ошибка "module 'sympy' has no attribute 'printing'" из-за конфликта версий
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "sympy"])

# Устанавливаем необходимые библиотеки
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "deep-translator"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "accelerate"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pillow"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas"])

print("\n✅ Все зависимости установлены!")
print("\n⚠️  ВАЖНО: теперь перезапустите среду выполнения!")
print("Меню: Runtime → Restart runtime (Среда выполнения → Перезапустить)")
print("После перезапуска запустите ВСЕ ячейки заново, начиная с этой.")


## ⚙️ Шаг 2: Инициализация модели

Загружаем Stable Diffusion модель (может занять 1-2 минуты в первый раз)

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image
import os
from pathlib import Path

# Проверяем, есть ли GPU (если в Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Используем: {device}")
if device == "cuda":
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
else:
    print("⚠️  GPU не найден! Включите его: Runtime → Change runtime type → GPU")
    print("   Без GPU генерация будет ОЧЕНЬ медленной.")

print("\n⏳ Загружаем БЫСТРУЮ модель SD-Turbo...")
print("(Первая загрузка: 1-2 минуты, потом быстрее)\n")

# SD-Turbo — специальная ускоренная модель.
# Генерирует картинку за 1 шаг вместо 25 — в 10-15 раз быстрее обычной Stable Diffusion,
# при этом качество вполне достаточное для иллюстраций к карточкам.
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    safety_checker=None
)

pipe = pipe.to(device)

if device == "cpu":
    pipe.enable_attention_slicing()
    print("\n💡 Используются оптимизации для CPU")
else:
    print("\n⚡ Используется GPU ускорение")

print("\n✅ Быстрая модель загружена и готова!")
print("💡 Генерация одной картинки теперь займет 1-5 секунд на GPU")


## 💾 Шаг 3: Подключаем Google Диск (рекомендуется при многих колодах)

**Зачем это нужно:** Colab стирает все загруженные файлы, когда вы закрываете вкладку или сессия отключается по неактивности. Если у вас десятки-сотни колод, лучше хранить исходные CSV и уже сгенерированные картинки на Google Диске — тогда при следующем запуске ничего не потеряется и не придётся рисовать картинки заново.

**Что нужно сделать один раз:**
1. Запустите ячейку ниже — появится запрос на доступ к вашему Google Диску, разрешите
2. В своём Google Диске создайте папку `FlashcardGenerator` (можно через приложение Диска на телефоне)
3. Внутри неё создайте папку `decks` — туда будете закидывать CSV файлы (можно с подпапками по языкам)

**Если у вас всего несколько колод** — можно пропустить этот шаг и просто загружать файлы в сессию Colab, как раньше.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Базовая папка на Google Диске — создайте её заранее (см. инструкцию выше)
DRIVE_BASE = '/content/drive/MyDrive/FlashcardGenerator'

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, 'decks'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, 'flashcard_images'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_BASE, 'output'), exist_ok=True)

print(f"✅ Google Диск подключён")
print(f"📁 Папка для CSV файлов: {DRIVE_BASE}/decks/")
print(f"📁 Папка для картинок (кэш): {DRIVE_BASE}/flashcard_images/")
print(f"📁 Папка для готовых HTML: {DRIVE_BASE}/output/")
print(f"\n💡 Загрузите CSV файлы в decks/ через приложение Google Диска на телефоне —")
print(f"   они появятся здесь автоматически, без ручной загрузки в Colab каждый раз.")

## 📥 Шаг 4: Загрузка карточек из NotebookLM

In [ ]:
import json
import csv
import os
from pathlib import Path
from typing import List, Dict

def load_flashcards_from_csv(file_path: str) -> List[Dict]:
    """
    Загружает карточки из ОДНОГО CSV файла в формате NotebookLM:
    БЕЗ заголовков, 2 столбца — текст на изучаемом языке, русский перевод.
    """
    flashcards = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            for row in reader:
                if len(row) >= 2:
                    front = row[0].strip()
                    back = row[1].strip()
                    if front and back:
                        flashcards.append({"question": front, "answer": back})
    except FileNotFoundError:
        print(f"❌ Файл не найден: {file_path}")
    return flashcards


def load_all_decks_grouped(decks_folder: str) -> Dict[str, Dict[str, List[Dict]]]:
    """
    Загружает МНОГО колод с группировкой по подпапкам — удобно для десятков/сотен колод.

    Структура папок:
        decks/
          Английский/
            Нейронаука.csv
            Базовые фразы.csv
          Арабский/
            Основы.csv
            Путешествия.csv
          Одиночный файл без подпапки.csv   ← попадёт в группу "Общее"

    Возвращает:
        {
            "Английский": {"Нейронаука": [...карточки...], "Базовые фразы": [...]},
            "Арабский": {"Основы": [...], "Путешествия": [...]},
            "Общее": {"Одиночный файл без подпапки": [...]}
        }

    Каждая группа (например, язык) станет ОТДЕЛЬНЫМ HTML файлом на выходе —
    так итоговые файлы остаются небольшими даже при сотнях колод суммарно,
    и список колод в боковом меню не разрастается до бесконечности.
    """
    grouped: Dict[str, Dict[str, List[Dict]]] = {}
    folder = Path(decks_folder)

    if not folder.exists():
        print(f"⚠️  Папка '{decks_folder}' не найдена.")
        return grouped

    # CSV файлы прямо в decks/ (без подпапки) — группа "Общее"
    root_csvs = sorted(folder.glob("*.csv"))
    if root_csvs:
        grouped["Общее"] = {}
        for csv_file in root_csvs:
            cards = load_flashcards_from_csv(str(csv_file))
            if cards:
                grouped["Общее"][csv_file.stem] = cards
                print(f"✅ [Общее] «{csv_file.stem}»: {len(cards)} карточек")

    # Подпапки — каждая подпапка = отдельная группа (например, язык)
    for subfolder in sorted(p for p in folder.iterdir() if p.is_dir()):
        group_name = subfolder.name
        csv_files = sorted(subfolder.glob("*.csv"))
        if not csv_files:
            continue
        grouped[group_name] = {}
        for csv_file in csv_files:
            cards = load_flashcards_from_csv(str(csv_file))
            if cards:
                grouped[group_name][csv_file.stem] = cards
                print(f"✅ [{group_name}] «{csv_file.stem}»: {len(cards)} карточек")

    return grouped


# ═══════════════════════════════════════════════════════════
# ВЫБЕРИТЕ ИСТОЧНИК ДАННЫХ:
# ═══════════════════════════════════════════════════════════

# Если подключали Google Диск (см. ячейку выше) — используйте эту папку:
try:
    decks_source = os.path.join(DRIVE_BASE, "decks")
except NameError:
    # Google Диск не подключали — используем обычную папку сессии Colab
    decks_source = "decks"

flashcards_grouped = load_all_decks_grouped(decks_source)

# Если ничего не нашлось — тестовые данные, чтобы можно было проверить работу
if not flashcards_grouped:
    print("\n⚠️  Колоды не найдены — используем тестовый набор для проверки")
    flashcards_grouped = {
        "Тест": {
            "Тестовая колода": [
                {"question": "Almost every drug of addiction hijacks that dopamine system.",
                 "answer": "Почти каждый вызывающий зависимость препарат захватывает дофаминовую систему."},
                {"question": "Dopamine is a key neurotransmitter, a key chemical in the brain.",
                 "answer": "Дофамин — это ключевой нейромедиатор, ключевое химическое вещество в мозге."}
            ]
        }
    }

total_decks = sum(len(decks) for decks in flashcards_grouped.values())
total_cards = sum(len(cards) for decks in flashcards_grouped.values() for cards in decks.values())
print(f"\n📚 Групп (языков/тем): {len(flashcards_grouped)}")
print(f"📚 Колод всего: {total_decks}")
print(f"🃏 Карточек всего: {total_cards}")
for group_name, decks in flashcards_grouped.items():
    print(f"\n   📁 {group_name}: {len(decks)} колод")
    for deck_name, cards in decks.items():
        print(f"      • {deck_name}: {len(cards)} карточек")


## 🎨 Шаг 5: Генерация промптов для картинок

In [ ]:
from deep_translator import GoogleTranslator

def translate_for_image_prompt(text: str) -> str:
    """
    Переводит текст карточки на английский — ТОЛЬКО для генерации картинки.
    Stable Diffusion гораздо лучше понимает английские описания,
    чем арабский, русский или другие языки.

    На самой карточке отображается оригинальный текст без изменений —
    перевод используется исключительно "за кулисами" для рисования.
    """
    try:
        # source='auto' — сам определит язык (арабский, русский и т.д.)
        translated = GoogleTranslator(source='auto', target='en').translate(text)
        return translated if translated else text
    except Exception as e:
        print(f"⚠️  Не удалось перевести для промпта, используем оригинал: {e}")
        return text


def create_smart_prompt(question: str, answer: str, topic: str = "") -> str:
    """
    Создает промпт для Stable Diffusion.

    Работает с любым языком карточек (английский, арабский, и т.д.) —
    сначала переводит фразу на английский для лучшего понимания моделью,
    затем определяет стиль (абстрактный/конкретный) уже по переводу.
    """

    clean_sentence = question.strip().strip('"')

    # Переводим на английский для промпта (модель рисует по-английски лучше всего)
    english_for_prompt = translate_for_image_prompt(clean_sentence)

    # Слова-маркеры абстрактных/эмоциональных понятий (проверяем по переводу)
    abstract_markers = [
        "apathy", "motivation", "desire", "happiness", "ambition",
        "identity", "self", "purpose", "value", "attention",
        "willing", "consciously", "subjective", "cost", "worth",
        "separate", "distinct", "equate", "resilient", "love",
        "faith", "belief", "soul", "patience", "gratitude"
    ]

    is_abstract = any(marker in english_for_prompt.lower() for marker in abstract_markers)

    if is_abstract:
        style = (
            "abstract conceptual illustration, symbolic metaphor, "
            "soft gradient background, minimalist digital art"
        )
    else:
        style = (
            "clear realistic illustration, simple scene, "
            "clean background, educational style"
        )

    prompt = f"""{english_for_prompt}, {style}, no text on image, no letters, no words, no captions, soft colors, high quality"""

    return prompt.strip()

# Тестируем на первых карточках первой колоды первой группы
print("📝 Примеры промтов (с автопереводом для картинки):\n")
first_group_name = list(flashcards_grouped.keys())[0]
first_deck_name = list(flashcards_grouped[first_group_name].keys())[0]
for card in flashcards_grouped[first_group_name][first_deck_name][:3]:
    prompt = create_smart_prompt(card['question'], card['answer'], first_deck_name)
    print(f"Оригинал: {card['question']}")
    print(f"Промт для AI: {prompt}\n")


## 🚀 Шаг 6: ГЕНЕРИРУЕМ КАРТИНКИ!

In [ ]:
import time

# ── Единая папка ПРИЛОЖЕНИЯ — здесь будут жить и картинки, и итоговый index.html ──
try:
    APP_DIR = os.path.join(DRIVE_BASE, "FlashcardApp")
except NameError:
    APP_DIR = "FlashcardApp"

base_output_dir = os.path.join(APP_DIR, "images")
Path(base_output_dir).mkdir(exist_ok=True, parents=True)

total_cards = sum(len(cards) for decks in flashcards_grouped.values() for cards in decks.values())
print("🎨 НАЧИНАЕМ ГЕНЕРАЦИЮ КАРТИНОК!\n")
print(f"📚 Всего карточек: {total_cards}")
print("⏳ SD-Turbo: примерно 1-5 секунд на картинку (на GPU)")
print("💾 Уже готовые картинки (из прошлых запусков) генерироваться заново не будут\n")

processed_count = 0
skipped_count = 0
generated_count = 0

# Настройки сжатия — важно при сотнях карточек, чтобы приложение оставалось лёгким
IMAGE_SIZE = 384
IMAGE_QUALITY = 80

for group_name, decks in flashcards_grouped.items():
    for deck_name, cards in decks.items():
        deck_dir = Path(base_output_dir) / group_name / deck_name
        deck_dir.mkdir(parents=True, exist_ok=True)

        print(f"\n{'#'*60}")
        print(f"📁 {group_name} / 📚 {deck_name} ({len(cards)} карточек)")
        print(f"{'#'*60}")

        for idx, card in enumerate(cards, 1):
            processed_count += 1
            image_path = str(deck_dir / f"card_{idx:03d}.jpg")

            if os.path.exists(image_path):
                card['image_path'] = image_path
                card['image_generated'] = True
                card['topic'] = deck_name
                skipped_count += 1
                print(f"⏭️  Карточка {idx}/{len(cards)} — картинка уже есть, пропускаем")
                continue

            print(f"\n🎴 Карточка {idx}/{len(cards)} (прогресс: {processed_count}/{total_cards})")
            print(f"📝 Текст: {card['question'][:60]}...")

            prompt = create_smart_prompt(card['question'], card['answer'], deck_name)
            print(f"⏳ Генерируем изображение...")

            try:
                start_time = time.time()

                with torch.no_grad():
                    image = pipe(
                        prompt,
                        height=IMAGE_SIZE,
                        width=IMAGE_SIZE,
                        num_inference_steps=1,
                        guidance_scale=0.0,
                    ).images[0]

                image = image.convert("RGB")
                image.save(image_path, "JPEG", quality=IMAGE_QUALITY, optimize=True)

                elapsed = time.time() - start_time
                file_size_kb = os.path.getsize(image_path) / 1024
                print(f"✅ Готово за {elapsed:.1f} сек, размер файла: {file_size_kb:.0f} КБ")

                card['image_path'] = image_path
                card['prompt'] = prompt
                card['image_generated'] = True
                card['topic'] = deck_name
                generated_count += 1

            except Exception as e:
                print(f"❌ Ошибка при генерации: {e}")
                card['image_generated'] = False
                card['topic'] = deck_name

print(f"\n\n{'='*60}")
print(f"🎉 ГЕНЕРАЦИЯ ЗАВЕРШЕНА!")
print(f"{'='*60}")
print(f"✅ Новых картинок создано: {generated_count}")
print(f"⏭️  Пропущено (уже были готовы): {skipped_count}")
print(f"📁 Все изображения в: {base_output_dir}/<группа>/<колода>/")


## 👀 Шаг 7: Просмотр карточек с картинками

In [ ]:
from IPython.display import display, HTML, Image as IPImage
import re

def is_rtl_text(text: str) -> bool:
    """Определяет, содержит ли текст арабские/ивритские символы (для направления письма)"""
    rtl_pattern = re.compile(r'[\u0591-\u07FF\uFB1D-\uFDFD\uFE70-\uFEFC]')
    return bool(rtl_pattern.search(text))

def display_beautiful_flashcard(card: Dict, card_number: int, total: int):
    """Красиво отображает карточку с изображением. Поддерживает RTL (арабский, иврит)."""
    
    image_html = ""
    if card.get('image_path') and os.path.exists(card['image_path']):
        image_html = f'<div style="text-align: center; margin: 20px 0;"><img src="{card["image_path"]}" style="max-width: 100%; max-height: 400px; border-radius: 15px; box-shadow: 0 10px 30px rgba(0,0,0,0.3);"></div>'
    
    text_dir = "rtl" if is_rtl_text(card['question']) else "ltr"
    font_size = "26px" if text_dir == "rtl" else "18px"
    
    html = f"""
    <div style="
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        border-radius: 20px;
        padding: 30px;
        margin: 20px 0;
        color: white;
        font-family: 'Segoe UI', Tahoma, sans-serif;
        box-shadow: 0 15px 40px rgba(0,0,0,0.3);
    ">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">
            <span style="background: rgba(255,255,255,0.2); padding: 8px 16px; border-radius: 20px; font-size: 13px;">
                {card.get('topic', 'General')}
            </span>
            <span style="font-size: 14px; opacity: 0.9;">Карточка {card_number}/{total}</span>
        </div>
        
        <div style="background: white; border-radius: 15px; padding: 20px; margin-bottom: 20px; color: #333;">
            <h3 style="margin: 0 0 10px 0; color: #667eea; font-size: 14px;">📖 ИЗУЧАЕМЫЙ ЯЗЫК</h3>
            <p dir="{text_dir}" style="margin: 0; font-size: {font_size}; font-weight: 500;">{card['question']}</p>
        </div>
        
        {image_html}
        
        <div style="background: rgba(255,255,255,0.95); border-radius: 15px; padding: 20px; color: #333;">
            <h3 style="margin: 0 0 10px 0; color: #667eea; font-size: 14px;">🇷🇺 ПЕРЕВОД</h3>
            <p style="margin: 0; font-size: 16px; line-height: 1.6;">{card['answer']}</p>
        </div>
    </div>
    """
    display(HTML(html))

# Показываем ПЕРВЫЕ 2 карточки из каждой колоды каждой группы —
# при сотнях карточек полный вывод был бы слишком длинным для превью
PREVIEW_LIMIT = 2

for group_name, decks in flashcards_grouped.items():
    print(f"\n{'='*20} 📁 ГРУППА: {group_name} {'='*20}")
    for deck_name, cards in decks.items():
        print(f"\n{'📚'*3} КОЛОДА: {deck_name} ({len(cards)} карточек, показаны первые {min(PREVIEW_LIMIT, len(cards))}) {'📚'*3}\n")
        for idx, card in enumerate(cards[:PREVIEW_LIMIT], 1):
            display_beautiful_flashcard(card, idx, len(cards))


## 💾 Шаг 8: Резервная копия данных

In [ ]:
# Резервная копия всех данных в JSON/CSV (на всякий случай, не используется приложением напрямую)
try:
    export_dir = os.path.join(DRIVE_BASE, "backup")
except NameError:
    export_dir = "backup"
os.makedirs(export_dir, exist_ok=True)

json_file = os.path.join(export_dir, "flashcards_all_groups.json")
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(flashcards_grouped, f, ensure_ascii=False, indent=2)
print(f"✅ Резервная копия JSON: {json_file}")

csv_file = os.path.join(export_dir, "flashcards_all_groups.csv")
all_rows = []
for group_name, decks in flashcards_grouped.items():
    for deck_name, cards in decks.items():
        for card in cards:
            row = dict(card)
            row['group'] = group_name
            row['deck'] = deck_name
            all_rows.append(row)

if all_rows:
    keys = list(all_rows[0].keys())
    with open(csv_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(all_rows)
    print(f"✅ Резервная копия CSV: {csv_file}")

print(f"\n📊 Итоговая статистика:")
for group_name, decks in flashcards_grouped.items():
    total_in_group = sum(len(cards) for cards in decks.values())
    print(f"   📁 {group_name}: {len(decks)} колод, {total_in_group} карточек")


## 🌐 Шаг 9: Собираем ОДНО приложение со всеми колодами

In [ ]:
import shutil

def create_full_app(flashcards_grouped: Dict[str, Dict[str, List[Dict]]], app_dir: str):
    """
    Собирает ОДНО приложение (index.html) со всеми группами и колодами сразу.

    В отличие от предыдущей версии, картинки НЕ встраиваются в текст HTML
    (base64) — они остаются обычными файлами в папке images/, а HTML
    ссылается на них по относительному пути. Это держит приложение лёгким
    даже при сотнях карточек с картинками.

    Структура на выходе:
        app_dir/
          index.html          ← открывать в Safari
          data/
            Английский.js     ← данные колод этой группы (текст, без картинок)
            Арабский.js
          images/
            Английский/Нейронаука/card_001.jpg
            ...
    """

    data_dir = os.path.join(app_dir, "data")
    os.makedirs(data_dir, exist_ok=True)

    def safe_filename(name: str) -> str:
        return re.sub(r'[\\/*?:"<>|]', '_', name).strip()

    # ── 1. Пишем по одному JS файлу с данными на каждую группу ──
    script_tags = []
    for group_name, decks in flashcards_grouped.items():
        decks_for_js = {}
        for deck_name, cards in decks.items():
            prepared_cards = []
            for card in cards:
                card_copy = {
                    "question": card["question"],
                    "answer": card["answer"],
                }
                if card.get("image_path") and os.path.exists(card["image_path"]):
                    # Относительный путь от index.html до картинки
                    rel_path = os.path.relpath(card["image_path"], app_dir).replace(os.sep, "/")
                    card_copy["image"] = rel_path
                prepared_cards.append(card_copy)
            decks_for_js[deck_name] = prepared_cards

        safe_group = safe_filename(group_name)
        js_filename = f"{safe_group}.js"
        js_path = os.path.join(data_dir, js_filename)

        js_content = (
            "window.ALL_DECKS = window.ALL_DECKS || {};\n"
            f"window.ALL_DECKS[{json.dumps(group_name)}] = {json.dumps(decks_for_js, ensure_ascii=False)};\n"
        )
        with open(js_path, "w", encoding="utf-8") as f:
            f.write(js_content)

        script_tags.append(f'<script src="data/{js_filename}"></script>')

    script_tags_html = "\n        ".join(script_tags)

    # ── 2. Пишем index.html — единое приложение со всеми группами ──
    html = f"""
    <!DOCTYPE html>
    <html lang="ru">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Мои флеш-карточки 🎓</title>
        <style>
            * {{ margin: 0; padding: 0; box-sizing: border-box; -webkit-tap-highlight-color: transparent; }}

            :root {{
                --bg: #131316;
                --card-bg: #26262b;
                --text-primary: #f2f2f4;
                --text-secondary: #9a9aa2;
                --accent-blue: #4d9dff;
                --accent-red: #ff5a52;
                --accent-green: #34d566;
                --divider: #3a3a40;
            }}

            body {{
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                background: var(--bg);
                min-height: 100vh;
                color: var(--text-primary);
            }}

            .app {{
                max-width: 640px;
                margin: 0 auto;
                padding: 14px 16px 24px;
                min-height: 100vh;
                display: flex;
                flex-direction: column;
            }}

            .top-bar {{ display: flex; align-items: center; gap: 12px; margin-bottom: 18px; }}

            .icon-btn {{
                background: none; border: none; color: var(--text-primary);
                font-size: 24px; width: 40px; height: 40px;
                display: flex; align-items: center; justify-content: center;
                cursor: pointer; border-radius: 50%; flex-shrink: 0; padding: 0;
            }}

            .icon-btn:active {{ background: rgba(255,255,255,0.08); }}

            .top-title {{
                flex: 1; text-align: center; font-size: 18px; font-weight: 700;
                white-space: nowrap; overflow: hidden; text-overflow: ellipsis; padding: 0 6px;
            }}

            .icon-spacer {{ width: 40px; flex-shrink: 0; }}

            .sidebar-overlay {{ position: fixed; inset: 0; background: rgba(0,0,0,0.6); z-index: 100; display: none; }}
            .sidebar-overlay.open {{ display: block; }}

            .sidebar {{
                position: fixed; top: 0; left: 0; bottom: 0; width: 300px; max-width: 86vw;
                background: #1c1c20; z-index: 101; transform: translateX(-100%);
                transition: transform 0.25s ease-out; overflow-y: auto;
                box-shadow: 4px 0 24px rgba(0,0,0,0.5);
            }}

            .sidebar.open {{ transform: translateX(0); }}

            .sidebar-header {{
                padding: 20px 16px; font-size: 16px; font-weight: 700;
                display: flex; justify-content: space-between; align-items: center;
                border-bottom: 1px solid var(--divider); position: sticky; top: 0;
                background: #1c1c20; z-index: 2;
            }}

            .sidebar-close {{
                background: rgba(255,255,255,0.08); border: none; color: var(--text-primary);
                width: 30px; height: 30px; border-radius: 8px; font-size: 16px; cursor: pointer;
            }}

            /* ── Группы (папки) ── */
            .group-header {{
                display: flex; align-items: center; gap: 8px;
                padding: 14px 16px; cursor: pointer;
                background: #202024; border-bottom: 1px solid var(--divider);
                font-weight: 700; font-size: 14px;
            }}

            .group-header:active {{ background: #26262c; }}

            .group-toggle-icon {{ color: var(--text-secondary); font-size: 11px; width: 12px; }}

            .group-count {{ color: var(--text-secondary); font-weight: 400; font-size: 12px; margin-left: auto; }}

            .group-decks {{ display: none; }}
            .group-decks.expanded {{ display: block; }}

            .deck-item {{
                display: flex; align-items: center; gap: 12px;
                padding: 12px 16px 12px 28px; cursor: pointer;
                border-bottom: 1px solid #232328;
            }}

            .deck-item:active, .deck-item.active {{ background: #232329; }}
            .deck-item.active {{ border-left: 3px solid var(--accent-blue); padding-left: 25px; }}

            .deck-thumb {{ width: 44px; height: 44px; border-radius: 10px; object-fit: cover; flex-shrink: 0; background: #333; }}
            .deck-thumb-placeholder {{
                width: 44px; height: 44px; border-radius: 10px; background: #34343c;
                display: flex; align-items: center; justify-content: center; font-size: 17px; flex-shrink: 0;
            }}

            .deck-info {{ flex: 1; min-width: 0; }}

            .deck-name-row {{ display: flex; align-items: center; gap: 6px; }}

            .deck-name {{
                font-size: 13.5px; font-weight: 600; color: var(--text-primary);
                white-space: nowrap; overflow: hidden; text-overflow: ellipsis; flex: 1; min-width: 0;
            }}

            .deck-rename-btn {{
                background: none; border: none; color: var(--text-secondary);
                font-size: 13px; padding: 4px 6px; cursor: pointer; flex-shrink: 0; border-radius: 6px;
            }}

            .deck-rename-btn:active {{ background: rgba(255,255,255,0.1); color: var(--text-primary); }}

            .deck-count {{ font-size: 11px; color: var(--text-secondary); margin-top: 2px; }}

            .deck-progress-bar {{
                display: flex; height: 5px; border-radius: 3px; overflow: hidden;
                background: #38383e; margin-top: 6px;
            }}

            .deck-progress-bar .seg.correct {{ background: var(--accent-green); }}
            .deck-progress-bar .seg.incorrect {{ background: var(--accent-red); }}
            .deck-progress-bar .seg.remaining {{ background: transparent; }}

            /* ── Карточка ── */
            .card {{
                background: var(--card-bg); border-radius: 28px; padding: 22px 22px 8px;
                flex: 1; display: flex; flex-direction: column; min-height: 400px;
                animation: fadeIn 0.35s ease-out;
            }}

            @keyframes fadeIn {{ from {{ opacity: 0; transform: translateY(6px); }} to {{ opacity: 1; transform: translateY(0); }} }}

            .card-top-row {{ display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px; }}
            .card-counter {{ font-size: 14px; color: var(--text-secondary); font-weight: 500; }}
            .card-menu-dots {{ color: var(--text-secondary); font-size: 20px; padding: 4px 8px; }}

            .card-body {{ flex: 1; display: flex; flex-direction: column; justify-content: center; padding: 20px 0; }}

            .question-text {{ font-size: 30px; font-weight: 500; line-height: 1.28; color: var(--text-primary); }}

            .image-wrap {{ text-align: center; margin: 18px 0 6px; }}
            .image-wrap img {{ max-width: 100%; max-height: 220px; border-radius: 16px; }}

            .answer-block {{
                margin-top: 22px; padding-top: 18px; border-top: 1px solid var(--divider);
                max-height: 0; overflow: hidden; opacity: 0; transition: all 0.3s ease;
            }}

            .answer-block.show {{ max-height: 400px; opacity: 1; }}

            .answer-label {{ font-size: 12px; text-transform: uppercase; letter-spacing: 0.5px; color: var(--text-secondary); margin-bottom: 8px; }}
            .answer-text {{ font-size: 22px; font-weight: 400; color: var(--text-primary); line-height: 1.4; }}

            .reveal-link {{ text-align: center; color: var(--text-secondary); font-size: 16px; padding: 16px 0 14px; cursor: pointer; user-select: none; }}
            .reveal-link:active {{ opacity: 0.6; }}

            .nav-row {{ display: flex; justify-content: center; align-items: center; gap: 18px; margin-top: 20px; }}

            .nav-circle {{
                background: #1c1c20; border: 1.5px solid var(--divider); color: var(--accent-blue);
                width: 60px; height: 60px; border-radius: 50%;
                display: flex; align-items: center; justify-content: center;
                font-size: 22px; cursor: pointer; flex-shrink: 0;
            }}

            .nav-circle:active {{ background: #29292f; }}

            .nav-circle.nav-wrong {{
                width: auto; min-width: 68px; padding: 0 16px; border-radius: 30px;
                gap: 6px; color: var(--accent-red); font-size: 18px; font-weight: 600;
            }}

            .nav-circle.nav-right {{
                width: auto; min-width: 68px; padding: 0 16px; border-radius: 30px;
                gap: 6px; color: var(--accent-green); font-size: 18px; font-weight: 600;
            }}

            .progress-bar {{ width: 100%; height: 3px; background: #2a2a30; border-radius: 2px; margin-top: 22px; }}
            .progress-fill {{ height: 100%; background: var(--accent-blue); border-radius: 2px; transition: width 0.3s ease; }}

            .keyboard-hint {{ text-align: center; color: #55555c; font-size: 11px; margin-top: 14px; }}

            .results-title {{ text-align: center; font-size: 20px; color: var(--text-primary); margin-bottom: 22px; font-weight: 700; }}
            .results-summary {{ display: flex; justify-content: center; gap: 34px; margin-bottom: 28px; }}
            .results-stat {{ text-align: center; }}
            .results-stat-number {{ font-size: 34px; font-weight: 700; }}
            .results-stat-label {{ font-size: 12px; color: var(--text-secondary); margin-top: 4px; }}
            .results-stat.correct .results-stat-number {{ color: var(--accent-green); }}
            .results-stat.incorrect .results-stat-number {{ color: var(--accent-red); }}

            .chart-bar-row {{ display: flex; align-items: center; gap: 12px; margin-bottom: 14px; }}
            .chart-bar-label {{ width: 80px; font-size: 13px; color: var(--text-secondary); text-align: right; flex-shrink: 0; }}
            .chart-bar-track {{ flex: 1; background: #2a2a30; border-radius: 10px; height: 24px; overflow: hidden; }}
            .chart-bar-fill {{
                height: 100%; border-radius: 10px; display: flex; align-items: center;
                justify-content: flex-end; padding-right: 10px; color: #0c0c0e;
                font-size: 12px; font-weight: 700; transition: width 0.6s ease; min-width: 28px;
            }}
            .chart-bar-fill.correct {{ background: var(--accent-green); }}
            .chart-bar-fill.incorrect {{ background: var(--accent-red); }}

            .results-list {{ margin-top: 18px; max-height: 200px; overflow-y: auto; border-top: 1px solid var(--divider); padding-top: 10px; }}
            .results-list-item {{ display: flex; align-items: center; gap: 8px; padding: 6px 0; font-size: 13px; color: var(--text-secondary); border-bottom: 1px solid #2a2a30; }}

            .reset-btn {{
                display: block; margin: 20px auto 0; background: var(--accent-blue); color: #0c0c0e;
                border: none; padding: 12px 28px; border-radius: 24px; font-size: 15px; font-weight: 700; cursor: pointer;
            }}

            .empty-state {{ text-align: center; color: var(--text-secondary); padding: 60px 20px; font-size: 15px; }}

            @media (max-width: 420px) {{
                .card {{ padding: 18px 16px 6px; min-height: 340px; }}
                .question-text {{ font-size: 24px; }}
                .answer-text {{ font-size: 18px; }}
                .nav-row {{ gap: 10px; }}
                .nav-circle {{ width: 52px; height: 52px; font-size: 19px; }}
                .nav-circle.nav-wrong, .nav-circle.nav-right {{ min-width: 58px; padding: 0 12px; font-size: 16px; }}
            }}
        </style>
    </head>
    <body>
        {script_tags_html}

        <div class="sidebar-overlay" id="sidebarOverlay" onclick="closeSidebar()"></div>
        <div class="sidebar" id="sidebar">
            <div class="sidebar-header">
                <span>📚 Мои колоды</span>
                <button class="sidebar-close" onclick="closeSidebar()">✕</button>
            </div>
            <div id="deckList"></div>
        </div>

        <div class="app">
            <div class="top-bar">
                <button class="icon-btn" onclick="openSidebar()">☰</button>
                <div class="top-title" id="deckTitleBar"></div>
                <div class="icon-spacer"></div>
            </div>

            <div class="card" id="flashcard"></div>
            <div class="nav-row" id="navRow"></div>
            <div class="progress-bar"><div class="progress-fill" id="progress"></div></div>
            <div class="keyboard-hint" id="hint">⌨️ Стрелки — навигация · SPACE — ответ · Y/N — оценка</div>
        </div>

        <script>
            const ALL_DECKS = window.ALL_DECKS || {{}};
            const groupNames = Object.keys(ALL_DECKS);

            let currentGroupName = null;
            let currentDeckName = null;
            let currentCard = 0;
            let answerShown = false;
            let deckResults = {{}};
            let expandedGroups = new Set();

            function deckKey(group, deck) {{ return group + '::' + deck; }}

            function getResults() {{
                return deckResults[deckKey(currentGroupName, currentDeckName)];
            }}

            function isRTL(text) {{
                const rtlChars = /[\\u0591-\\u07FF\\uFB1D-\\uFDFD\\uFE70-\\uFEFC]/;
                return rtlChars.test(text);
            }}

            function currentDeck() {{
                return ALL_DECKS[currentGroupName][currentDeckName];
            }}

            const DECK_NAMES_STORAGE_KEY = 'flashcard_deck_display_names_v2';

            function loadDeckDisplayNames() {{
                try {{
                    const raw = localStorage.getItem(DECK_NAMES_STORAGE_KEY);
                    return raw ? JSON.parse(raw) : {{}};
                }} catch (e) {{ return {{}}; }}
            }}

            function saveDeckDisplayNames(obj) {{
                try {{ localStorage.setItem(DECK_NAMES_STORAGE_KEY, JSON.stringify(obj)); }} catch (e) {{}}
            }}

            let deckDisplayNames = loadDeckDisplayNames();

            function getDisplayName(key) {{ return deckDisplayNames[key] || key.split('::')[1]; }}

            function startRenameDeck(group, deck) {{
                const key = deckKey(group, deck);
                const current = getDisplayName(key);
                const newName = prompt('Новое название колоды:', current);
                if (newName && newName.trim() && newName.trim() !== current) {{
                    deckDisplayNames[key] = newName.trim();
                    saveDeckDisplayNames(deckDisplayNames);
                    renderSidebar();
                    if (currentGroupName === group && currentDeckName === deck) renderCard();
                }}
            }}

            function toggleGroup(group) {{
                if (expandedGroups.has(group)) expandedGroups.delete(group);
                else expandedGroups.add(group);
                renderSidebar();
            }}

            function renderSidebar() {{
                const listEl = document.getElementById('deckList');

                if (groupNames.length === 0) {{
                    listEl.innerHTML = '<div class="empty-state">Колоды не найдены</div>';
                    return;
                }}

                listEl.innerHTML = groupNames.map(group => {{
                    const decks = ALL_DECKS[group];
                    const deckNamesInGroup = Object.keys(decks);
                    const isExpanded = expandedGroups.has(group);

                    const decksHtml = deckNamesInGroup.map(deck => {{
                        const cards = decks[deck];
                        const key = deckKey(group, deck);
                        const firstImg = cards.find(c => c.image);
                        const thumb = firstImg
                            ? `<img class="deck-thumb" src="${{firstImg.image}}" alt="">`
                            : `<div class="deck-thumb-placeholder">📖</div>`;
                        const activeClass = (group === currentGroupName && deck === currentDeckName) ? 'active' : '';

                        const arr = deckResults[key] || new Array(cards.length).fill(null);
                        const total = cards.length;
                        const correct = arr.filter(r => r === true).length;
                        const incorrect = arr.filter(r => r === false).length;
                        const answered = correct + incorrect;
                        const remaining = total - answered;

                        let progressText;
                        if (answered === 0) progressText = `${{total}} карточек`;
                        else if (answered === total) progressText = `✅ Пройдено: ${{correct}}/${{total}}`;
                        else progressText = `${{answered}}/${{total}} пройдено`;

                        const displayName = getDisplayName(key);
                        const safeGroup = group.replace(/'/g, "\\\\'");
                        const safeDeck = deck.replace(/'/g, "\\\\'");

                        return `
                            <div class="deck-item ${{activeClass}}" onclick="selectDeck('${{safeGroup}}', '${{safeDeck}}')">
                                ${{thumb}}
                                <div class="deck-info">
                                    <div class="deck-name-row">
                                        <div class="deck-name">${{displayName}}</div>
                                        <button class="deck-rename-btn" onclick="event.stopPropagation(); startRenameDeck('${{safeGroup}}', '${{safeDeck}}')">✎</button>
                                    </div>
                                    <div class="deck-count">${{progressText}}</div>
                                    <div class="deck-progress-bar">
                                        <div class="seg correct" style="flex:${{correct}} ${{correct}} 0"></div>
                                        <div class="seg incorrect" style="flex:${{incorrect}} ${{incorrect}} 0"></div>
                                        <div class="seg remaining" style="flex:${{remaining}} ${{remaining}} 0"></div>
                                    </div>
                                </div>
                            </div>
                        `;
                    }}).join('');

                    const safeGroupId = group.replace(/[^a-zA-Z0-9]/g, '_');

                    return `
                        <div class="group-section">
                            <div class="group-header" onclick="toggleGroup('${{group.replace(/'/g, "\\\\'")}}')">
                                <span class="group-toggle-icon">${{isExpanded ? '▾' : '▸'}}</span>
                                <span>📁 ${{group}}</span>
                                <span class="group-count">${{deckNamesInGroup.length}} колод</span>
                            </div>
                            <div class="group-decks ${{isExpanded ? 'expanded' : ''}}" id="group-${{safeGroupId}}">
                                ${{decksHtml}}
                            </div>
                        </div>
                    `;
                }}).join('');
            }}

            function openSidebar() {{
                document.getElementById('sidebar').classList.add('open');
                document.getElementById('sidebarOverlay').classList.add('open');
            }}

            function closeSidebar() {{
                document.getElementById('sidebar').classList.remove('open');
                document.getElementById('sidebarOverlay').classList.remove('open');
            }}

            function selectDeck(group, deck) {{
                currentGroupName = group;
                currentDeckName = deck;
                expandedGroups.add(group);

                const key = deckKey(group, deck);
                if (!deckResults[key]) {{
                    deckResults[key] = new Array(ALL_DECKS[group][deck].length).fill(null);
                }}

                const arr = deckResults[key];
                const firstUnanswered = arr.findIndex(r => r === null);

                answerShown = false;
                closeSidebar();
                renderSidebar();

                if (firstUnanswered === -1 && arr.length > 0) {{
                    currentCard = arr.length - 1;
                    showResults();
                }} else {{
                    currentCard = firstUnanswered === -1 ? 0 : firstUnanswered;
                    document.getElementById('hint').style.display = 'block';
                    renderCard();
                }}
            }}

            function renderCard() {{
                const deck = currentDeck();
                const card = deck[currentCard];
                const cardEl = document.getElementById('flashcard');

                document.getElementById('deckTitleBar').textContent = getDisplayName(deckKey(currentGroupName, currentDeckName));

                let imageHtml = '';
                if (card.image) {{
                    imageHtml = `<div class="image-wrap"><img src="${{card.image}}" alt=""></div>`;
                }}

                const textDirection = isRTL(card.question) ? 'rtl' : 'ltr';

                cardEl.innerHTML = `
                    <div class="card-top-row">
                        <span class="card-counter">${{currentCard + 1}} из ${{deck.length}}</span>
                        <span class="card-menu-dots">⋮</span>
                    </div>
                    <div class="card-body">
                        <p class="question-text" dir="${{textDirection}}">${{card.question}}</p>
                        ${{imageHtml}}
                        <div class="answer-block" id="answerBlock">
                            <div class="answer-label">Перевод</div>
                            <div class="answer-text">${{card.answer}}</div>
                        </div>
                    </div>
                    <div class="reveal-link" onclick="toggleAnswer()">${{answerShown ? 'Скрыть ответ' : 'Показать ответ'}}</div>
                `;

                if (answerShown) cardEl.querySelector('#answerBlock').classList.add('show');

                document.getElementById('progress').style.width = (((currentCard + 1) / deck.length) * 100) + '%';
                renderNavRow();
            }}

            function renderNavRow() {{
                const deck = currentDeck();
                const arr = getResults();
                const correct = arr.filter(r => r === true).length;
                const incorrect = arr.filter(r => r === false).length;
                const isLast = currentCard === deck.length - 1;

                document.getElementById('navRow').innerHTML = `
                    <button class="nav-circle nav-prev" onclick="previousCard()" ${{currentCard === 0 ? 'style="opacity:0.35;pointer-events:none;"' : ''}}>←</button>
                    <button class="nav-circle nav-wrong" onclick="rate(false)"><span>✕</span><span>${{incorrect}}</span></button>
                    <button class="nav-circle nav-right" onclick="rate(true)"><span>✓</span><span>${{correct}}</span></button>
                    <button class="nav-circle nav-next" onclick="goNext()">${{isLast ? '🏁' : '→'}}</button>
                `;
            }}

            function toggleAnswer() {{ answerShown = !answerShown; renderCard(); }}

            function rate(knewIt) {{
                getResults()[currentCard] = knewIt;
                renderSidebar();
                goNext();
            }}

            function goNext() {{
                const deck = currentDeck();
                if (currentCard < deck.length - 1) {{
                    currentCard++;
                    answerShown = false;
                    renderCard();
                }} else {{
                    showResults();
                }}
            }}

            function previousCard() {{
                if (currentCard > 0) {{
                    currentCard--;
                    answerShown = false;
                    renderCard();
                }}
            }}

            function showResults() {{
                const deck = currentDeck();
                const arr = getResults();
                const correct = arr.filter(r => r === true).length;
                const incorrect = arr.filter(r => r === false).length;
                const total = deck.length;
                const correctPct = total > 0 ? Math.round((correct / total) * 100) : 0;
                const incorrectPct = total > 0 ? Math.round((incorrect / total) * 100) : 0;

                const listHtml = deck.map((card, idx) => {{
                    const icon = arr[idx] === true ? '✅' : (arr[idx] === false ? '❌' : '⏭️');
                    return `<div class="results-list-item">${{icon}} <span>${{card.question}}</span></div>`;
                }}).join('');

                document.getElementById('flashcard').innerHTML = `
                    <div style="padding-top: 10px;">
                        <div class="results-title">🎉 Колода «${{getDisplayName(deckKey(currentGroupName, currentDeckName))}}» пройдена!</div>
                        <div class="results-summary">
                            <div class="results-stat correct"><div class="results-stat-number">${{correct}}</div><div class="results-stat-label">✓ Знал</div></div>
                            <div class="results-stat incorrect"><div class="results-stat-number">${{incorrect}}</div><div class="results-stat-label">✕ Не знал</div></div>
                        </div>
                        <div class="chart-bar-row">
                            <div class="chart-bar-label">Знал</div>
                            <div class="chart-bar-track"><div class="chart-bar-fill correct" style="width: ${{Math.max(correctPct, correct > 0 ? 8 : 0)}}%">${{correct > 0 ? correctPct + '%' : ''}}</div></div>
                        </div>
                        <div class="chart-bar-row">
                            <div class="chart-bar-label">Не знал</div>
                            <div class="chart-bar-track"><div class="chart-bar-fill incorrect" style="width: ${{Math.max(incorrectPct, incorrect > 0 ? 8 : 0)}}%">${{incorrect > 0 ? incorrectPct + '%' : ''}}</div></div>
                        </div>
                        <div class="results-list">${{listHtml}}</div>
                        <button class="reset-btn" onclick="resetDeck()">🔄 Пройти заново</button>
                    </div>
                `;

                document.getElementById('navRow').innerHTML = '';
                document.getElementById('progress').style.width = '100%';
                document.getElementById('hint').style.display = 'none';
            }}

            function resetDeck() {{
                currentCard = 0;
                answerShown = false;
                deckResults[deckKey(currentGroupName, currentDeckName)] = new Array(currentDeck().length).fill(null);
                document.getElementById('hint').style.display = 'block';
                renderSidebar();
                renderCard();
            }}

            document.addEventListener('keydown', (e) => {{
                if (e.key === ' ') {{ e.preventDefault(); toggleAnswer(); }}
                if (e.key === 'ArrowRight') goNext();
                if (e.key === 'ArrowLeft') previousCard();
                if (e.key === 'y' || e.key === 'Y') rate(true);
                if (e.key === 'n' || e.key === 'N') rate(false);
            }});

            // ── Инициализация ──
            groupNames.forEach(group => {{
                Object.keys(ALL_DECKS[group]).forEach(deck => {{
                    deckResults[deckKey(group, deck)] = new Array(ALL_DECKS[group][deck].length).fill(null);
                }});
            }});

            if (groupNames.length > 0) {{
                currentGroupName = groupNames[0];
                currentDeckName = Object.keys(ALL_DECKS[currentGroupName])[0];
                expandedGroups.add(currentGroupName);
                renderSidebar();
                renderCard();
            }} else {{
                document.getElementById('flashcard').innerHTML = '<div class="empty-state">Колоды не найдены.<br>Добавьте CSV файлы в decks/ и запустите ноутбук заново.</div>';
                document.getElementById('hint').style.display = 'none';
            }}
        </script>
    </body>
    </html>
    """

    with open(os.path.join(app_dir, "index.html"), "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Приложение собрано: {os.path.join(app_dir, 'index.html')}")
    print(f"📁 Данных колод: {len(flashcards_grouped)} групп")

    return os.path.join(app_dir, "index.html")


# ── Собираем приложение ──
app_index_path = create_full_app(flashcards_grouped, APP_DIR)

# ── Упаковываем всю папку приложения в один ZIP для удобной передачи на телефон ──
zip_base_path = os.path.join(os.path.dirname(APP_DIR) or ".", "FlashcardApp")
zip_path = shutil.make_archive(zip_base_path, "zip", APP_DIR)

size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"\n📦 Архив готов: {zip_path} ({size_mb:.1f} МБ)")
print(f"👉 Скачайте этот ZIP, распакуйте на телефоне, откройте index.html внутри распакованной папки")


## 📚 ИНСТРУКЦИЯ ДЛЯ ПОЛЬЗОВАТЕЛЯ

In [ ]:
instructions = """
╔══════════════════════════════════════════════════════════════════════╗
║          🎓 ВАШЕ ПРИЛОЖЕНИЕ СО ВСЕМИ КОЛОДАМИ ГОТОВО! 🎓            ║
╚══════════════════════════════════════════════════════════════════════╝

✅ ЧТО БЫЛО СОЗДАНО — ОДНО ПРИЛОЖЕНИЕ СО ВСЕМИ КОЛОДАМИ СРАЗУ:

📦 FlashcardApp.zip   ← скачайте именно этот архив
   └── при распаковке:
       index.html      ← ЭТОТ файл открывать в Safari
       data/*.js        ← данные всех колод (текст)
       images/           ← картинки отдельными файлами

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📁 СТРУКТУРА КОЛОД (без ограничений по количеству):

decks/
  Английский/
    Нейронаука.csv
    Базовые фразы.csv
  Арабский/
    Основы.csv
  Испанский/
    Ресторан.csv
    ... сколько угодно ...

Каждая ПОДПАПКА — это раздел (📁) в боковом меню приложения.
Внутри раздела — все ваши колоды этого языка/темы.
ВСЁ это — в ОДНОМ index.html, никаких отдельных файлов на каждый язык.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📲 КАК ОТКРЫТЬ НА ТЕЛЕФОНЕ (один раз):

1. Скачайте FlashcardApp.zip на iPhone
2. В приложении "Файлы" нажмите на архив → "Unzip" (распаковать)
3. Откройте появившуюся папку FlashcardApp → нажмите на index.html
4. Если открылся простой просмотрщик (пустая карточка) — нажмите
   "Поделиться" → "Открыть в Safari"
5. Готово — все ваши колоды доступны через значок ☰ слева вверху,
   сгруппированные по папкам (языкам/темам)

💡 Сохраните ЗНАЧОК Safari на эту страницу на домашний экран
   (Поделиться → «На экран Домой») — будет как обычное приложение

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🔄 КОГДА ДОБАВЛЯЕТЕ НОВЫЕ КОЛОДЫ:

1. Кладёте новые CSV в соответствующую подпапку decks/ (на Google Диске)
2. Запускаете ноутбук заново (Runtime → Run all)
   — старые картинки НЕ перегенерируются (кэш), только новые карточки
3. Скачиваете обновлённый FlashcardApp.zip, распаковываете поверх старой
   папки на телефоне (замените старые файлы новыми)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎯 КАК ПОЛЬЗОВАТЬСЯ ПРИЛОЖЕНИЕМ:

1. ☰ открывает меню — там разделы (📁) по языкам/темам
2. Нажмите на раздел — развернётся список его колод (с превью и прогрессом)
3. Выберите колоду — начнётся показ карточек
4. "Показать ответ" → кнопки ✕ (не знал) / ✓ (знал) сразу листают дальше
5. Кнопка "→" — переход без оценки; в конце — диаграмма результатов
6. ✎ рядом с названием колоды — переименовать (сохраняется в этом браузере)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

⌨️  ГОРЯЧИЕ КЛАВИШИ:  → следующая · ← предыдущая · SPACE ответ · Y/N оценка

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 НАДОЕЛО КАЖДЫЙ РАЗ СКАЧИВАТЬ ZIP?

Прокрутите ноутбук до самого низа (Шаг 10) — там есть настройка
публикации по ПОСТОЯННОЙ ссылке через GitHub Pages. Один раз
настраиваете (5 минут) — и дальше просто открываете сохранённую
ссылку в Safari, без скачивания и распаковки zip каждый раз.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📦 ЗАГРУЗКА МНОГИХ КОЛОД СРАЗУ:

Можно закинуть сразу десятки CSV файлов одним действием — в приложении
Google Диска на телефоне выделите сразу все файлы (долгое нажатие →
отметить остальные) и переместите их все разом в нужные подпапки
decks/<Язык>/. Colab увидит их все при следующем запуске.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🎉 ГОТОВО! ВСЕ ВАШИ КОЛОДЫ — В ОДНОМ ПРИЛОЖЕНИИ! 🎉
"""

print(instructions)

## 🌍 Шаг 10 (опционально, но рекомендуется): публикуем по постоянной ссылке

**Проблема, которую это решает:** без этого шага каждый раз при добавлении новой колоды приходится заново скачивать `FlashcardApp.zip` на телефон, распаковывать и открывать заново.

**С этим шагом:** один раз настраиваете — и дальше просто открываете **одну и ту же сохранённую ссылку** в Safari. При каждом обновлении колод вы просто запускаете эту ячейку заново — через минуту ссылка сама покажет актуальные данные. Скачивать и распаковывать zip больше не нужно никогда.

**Что нужно подготовить один раз (5 минут):**
1. Если нет аккаунта — зарегистрируйтесь на [github.com](https://github.com) (бесплатно)
2. Создайте новый репозиторий: кнопка **New** → назовите, например, `my-flashcards` → **обязательно Public** (для бесплатного хостинга) → Create repository
3. Зайдите в **Settings** этого репозитория → **Pages** (слева в меню) → в разделе Source выберите **Deploy from a branch** → ветка **main**, папка **/(root)** → Save
4. Создайте токен доступа: **github.com → фото профиля → Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token** → отметьте галочку **repo** → Generate → скопируйте токен (он показывается только один раз!)

⚠️ Токен — это как пароль, никому его не показывайте. В ячейке ниже он вводится скрыто (звёздочками) и никуда не сохраняется.

In [ ]:
import subprocess
import shutil
from getpass import getpass

GITHUB_USERNAME = input("Ваш логин на GitHub: ").strip()
GITHUB_REPO = input("Название репозитория (например, my-flashcards): ").strip()
GITHUB_TOKEN = getpass("Personal Access Token (вводится скрыто): ").strip()

repo_dir = "/content/github_repo"
repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git"

if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

print("📥 Клонируем репозиторий...")
clone_result = subprocess.run(["git", "clone", repo_url, repo_dir], capture_output=True, text=True)
if clone_result.returncode != 0:
    print("❌ Не удалось клонировать репозиторий. Проверьте логин, название репозитория и токен.")
    print(clone_result.stderr)
else:
    subprocess.run(["git", "-C", repo_dir, "config", "user.email", "flashcards@generator.local"])
    subprocess.run(["git", "-C", repo_dir, "config", "user.name", "Flashcard Generator"])

    print("📋 Копируем приложение в репозиторий...")
    for item in os.listdir(APP_DIR):
        src = os.path.join(APP_DIR, item)
        dst = os.path.join(repo_dir, item)
        if os.path.isdir(dst):
            shutil.rmtree(dst)
        elif os.path.exists(dst):
            os.remove(dst)
        if os.path.isdir(src):
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

    subprocess.run(["git", "-C", repo_dir, "add", "."])
    commit_result = subprocess.run(
        ["git", "-C", repo_dir, "commit", "-m", "Обновление колод"],
        capture_output=True, text=True
    )

    if "nothing to commit" in commit_result.stdout:
        print("ℹ️  Изменений нет — публиковать нечего (данные не менялись с прошлого раза)")
    else:
        print("📤 Публикуем...")
        push_result = subprocess.run(["git", "-C", repo_dir, "push"], capture_output=True, text=True)
        if push_result.returncode != 0:
            print("❌ Ошибка при публикации:")
            print(push_result.stderr)
        else:
            print("✅ Опубликовано!")

    print(f"\n🌐 Ваше приложение (обновится в течение 1-2 минут):")
    print(f"   https://{GITHUB_USERNAME}.github.io/{GITHUB_REPO}/")
    print(f"\n💡 Сохраните эту ссылку на домашний экран iPhone (Поделиться → На экран Домой).")
    print(f"   В следующий раз просто запустите эту ячейку заново после добавления новых колод —")
    print(f"   скачивать и распаковывать zip больше не понадобится.")